# Retail Inventory Analytics — Data Cleaning & Understanding

## Project Overview

This project analyzes retail inventory data to identify inventory inefficiencies,
products and stores at risk, and actionable replenishment opportunities.

The analysis follows:

**Data Understanding → Data Cleaning → SQL Analysis → Inventory KPIs → 
Visualization → Demand Analysis → Business Recommendations**

This notebook focuses on **data understanding, quality validation, and cleaning**
before loading the data into a SQL database.

## Business Question

> Where are the biggest inventory inefficiencies, which products/stores are at
> risk, and what inventory actions should the business take?

## Dataset

The dataset contains daily store-product observations with information about:

- Inventory levels
- Units sold
- Units ordered
- Demand forecasts
- Pricing and discounts
- Weather conditions
- Holidays/promotions
- Seasonality

The dataset contains **73,100 records and 15 columns** covering
**2022-01-01 to 2024-01-01**.

## Important Limitations

The dataset does not contain:

- Supplier information
- Supplier lead time
- Warehouse information
- Purchase cost / COGS
- Explicit stockout flags
- Historical stock receipt dates

Therefore, inventory risk metrics will be treated as **indicators/proxies**
rather than direct measurements of actual stockouts or financial inventory
turnover.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:

DATA_PATH = Path("../data/raw/retail_store_inventory.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [4]:
df.shape

(73100, 15)

In [5]:
df.columns.tolist()

['Date',
 'Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Sold',
 'Units Ordered',
 'Demand Forecast',
 'Price',
 'Discount',
 'Weather Condition',
 'Holiday/Promotion',
 'Competitor Pricing',
 'Seasonality']

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
dtypes: float64(3), int64(5), object(7)
memory usage: 8.4+

In [7]:
df.describe()

,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing
count,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,0.497305,55.146077
std,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,0.499996,26.191408
min,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,0.000000,5.030000
25%,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000
50%,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000
75%,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000
max,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000


In [8]:
df.describe(include="object")

,Date,Store ID,Product ID,Category,Region,Weather Condition,Seasonality
count,73100,73100,73100,73100,73100,73100,73100
unique,731,5,20,5,4,4,4
top,2022-01-01,S001,P0001,Furniture,East,Sunny,Spring
freq,100,14620,3655,14699,18349,18290,18317


In [11]:
df.dtypes

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition      object
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality            object
dtype: object

In [12]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

In [13]:
df.dtypes

Date                  datetime64[ns]
Store ID                      object
Product ID                    object
Category                      object
Region                        object
Inventory Level                int64
Units Sold                     int64
Units Ordered                  int64
Demand Forecast              float64
Price                        float64
Discount                       int64
Weather Condition             object
Holiday/Promotion              int64
Competitor Pricing           float64
Seasonality                   object
dtype: object

In [14]:
print("Invalid dates:", df["Date"].isna().sum())

Invalid dates: 0


In [15]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing.sort_values("missing_count", ascending=False)

,missing_count,missing_percentage
Date,0,0.0
Store ID,0,0.0
Product ID,0,0.0
Category,0,0.0
Region,0,0.0
Inventory Level,0,0.0
Units Sold,0,0.0
Units Ordered,0,0.0
Demand Forecast,0,0.0
Price,0,0.0


In [17]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")

Duplicate rows: 0


In [19]:
key_columns = ["Date", "Store ID", "Product ID"]

duplicate_keys = df.duplicated(
    subset=key_columns,
    keep=False
).sum()

print(f"Duplicate Date + Store ID + Product ID combinations: {duplicate_keys:,}")

Duplicate Date + Store ID + Product ID combinations: 0


In [21]:
cardinality = {
    "Stores": df["Store ID"].nunique(),
    "Products": df["Product ID"].nunique(),
    "Categories": df["Category"].nunique(),
    "Regions": df["Region"].nunique()
}

pd.Series(cardinality)

Stores         5
Products      20
Categories     5
Regions        4
dtype: int64

In [22]:
for col in ["Store ID", "Product ID", "Category", "Region"]:
    print(f"\n{col}:")
    print(df[col].unique())


Store ID:
['S001' 'S002' 'S003' 'S004' 'S005']

Product ID:
['P0001' 'P0002' 'P0003' 'P0004' 'P0005' 'P0006' 'P0007' 'P0008' 'P0009'
 'P0010' 'P0011' 'P0012' 'P0013' 'P0014' 'P0015' 'P0016' 'P0017' 'P0018'
 'P0019' 'P0020']

Category:
['Groceries' 'Toys' 'Electronics' 'Furniture' 'Clothing']

Region:
['North' 'South' 'West' 'East']


In [23]:
print("Minimum date:", df["Date"].min())
print("Maximum date:", df["Date"].max())

Minimum date: 2022-01-01 00:00:00
Maximum date: 2024-01-01 00:00:00


In [24]:
print("Number of unique dates:", df["Date"].nunique())

Number of unique dates: 731


In [25]:
df["Date"].sort_values().unique()[:10]

<DatetimeArray>
['2022-01-01 00:00:00', '2022-01-02 00:00:00', '2022-01-03 00:00:00',
 '2022-01-04 00:00:00', '2022-01-05 00:00:00', '2022-01-06 00:00:00',
 '2022-01-07 00:00:00', '2022-01-08 00:00:00', '2022-01-09 00:00:00',
 '2022-01-10 00:00:00']
Length: 10, dtype: datetime64[ns]

### Numerical Data Validation

We now validate the numerical variables for:

- Negative values
- Zero values where relevant
- Unexpected ranges
- Potential outliers

Special attention is given to inventory, sales, orders, demand forecasts,
discounts, and pricing because these variables will be used in later
inventory and demand analysis.

In [26]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

print("Numerical columns:")
print(numeric_columns)

Numerical columns:
['Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Holiday/Promotion', 'Competitor Pricing']


In [27]:
negative_counts = (df[numeric_columns] < 0).sum()

negative_counts

Inventory Level         0
Units Sold              0
Units Ordered           0
Demand Forecast       673
Price                   0
Discount                0
Holiday/Promotion       0
Competitor Pricing      0
dtype: int64

In [28]:
numeric_summary = df[numeric_columns].agg(["min", "max", "mean", "median"]).T

numeric_summary

,min,max,mean,median
Inventory Level,50.00,500.00,274.469877,273.000
Units Sold,0.00,499.00,136.464870,107.000
Units Ordered,20.00,200.00,110.004473,110.000
Demand Forecast,-9.99,518.55,141.494720,113.015
Price,10.00,100.00,55.135108,55.050
Discount,0.00,20.00,10.009508,10.000
Holiday/Promotion,0.00,1.00,0.497305,0.000
Competitor Pricing,5.03,104.94,55.146077,55.010


In [29]:
zero_counts = (df[numeric_columns] == 0).sum()

zero_counts

Inventory Level           0
Units Sold              360
Units Ordered             0
Demand Forecast           0
Price                     0
Discount              14662
Holiday/Promotion     36747
Competitor Pricing        0
dtype: int64

In [30]:
df["Demand Forecast"].describe()

count    73100.000000
mean       141.494720
std        109.254076
min         -9.990000
25%         53.670000
50%        113.015000
75%        208.052500
max        518.550000
Name: Demand Forecast, dtype: float64

In [31]:
df[df["Demand Forecast"] < 0]["Demand Forecast"].head()

63    -2.40
141   -3.40
278   -3.91
511   -8.37
730   -2.99
Name: Demand Forecast, dtype: float64

In [32]:
print(
    "Negative Demand Forecast values:",
    (df["Demand Forecast"] < 0).sum()
)

Negative Demand Forecast values: 673


In [33]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts())


Store ID
Store ID
S001    14620
S002    14620
S003    14620
S004    14620
S005    14620
Name: count, dtype: int64

Product ID
Product ID
P0001    3655
P0002    3655
P0019    3655
P0018    3655
P0017    3655
P0016    3655
P0015    3655
P0014    3655
P0013    3655
P0012    3655
P0011    3655
P0010    3655
P0009    3655
P0008    3655
P0007    3655
P0006    3655
P0005    3655
P0004    3655
P0003    3655
P0020    3655
Name: count, dtype: int64

Category
Category
Furniture      14699
Toys           14643
Clothing       14626
Groceries      14611
Electronics    14521
Name: count, dtype: int64

Region
Region
East     18349
South    18297
North    18228
West     18226
Name: count, dtype: int64

Weather Condition
Weather Condition
Sunny     18290
Rainy     18278
Snowy     18272
Cloudy    18260
Name: count, dtype: int64

Seasonality
Seasonality
Spring    18317
Summer    18305
Winter    18285
Autumn    18193
Name: count, dtype: int64


In [34]:
print("Discount values:")
print(sorted(df["Discount"].unique()))

print("\nHoliday/Promotion values:")
print(sorted(df["Holiday/Promotion"].unique()))

Discount values:
[0, 5, 10, 15, 20]

Holiday/Promotion values:
[0, 1]


In [35]:
outlier_summary = []

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outliers,
        "Outlier %": outliers / len(df) * 100
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

,Column,Q1,Q3,Lower Bound,Upper Bound,Outlier Count,Outlier %
0,Inventory Level,162.00,387.0000,-175.50000,724.50000,0,0.000000
1,Units Sold,49.00,203.0000,-182.00000,434.00000,715,0.978112
2,Units Ordered,65.00,155.0000,-70.00000,290.00000,0,0.000000
3,Demand Forecast,53.67,208.0525,-177.90375,439.62625,732,1.001368
4,Price,32.65,77.8600,-35.16500,145.67500,0,0.000000
5,Discount,5.00,15.0000,-10.00000,30.00000,0,0.000000
6,Holiday/Promotion,0.00,1.0000,-1.50000,2.50000,0,0.000000
7,Competitor Pricing,32.68,77.8200,-35.03000,145.53000,0,0.000000


### Investigating Negative Demand Forecasts

Demand Forecast represents expected demand and therefore should not contain
negative values.

There are 673 observations with negative demand forecasts. Before deciding
how to treat them, we inspect their distribution and relationship with other
inventory variables.

In [36]:
negative_forecast = df[df["Demand Forecast"] < 0].copy()

print(f"Negative forecast records: {len(negative_forecast):,}")

negative_forecast["Demand Forecast"].describe()

Negative forecast records: 673


count    673.000000
mean      -3.693730
std        2.535978
min       -9.990000
25%       -5.470000
50%       -3.380000
75%       -1.540000
max       -0.010000
Name: Demand Forecast, dtype: float64

In [37]:
negative_forecast[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Category",
        "Inventory Level",
        "Units Sold",
        "Units Ordered",
        "Demand Forecast"
    ]
].head(20)

,Date,Store ID,Product ID,Category,Inventory Level,Units Sold,Units Ordered,Demand Forecast
63,2022-01-01,S004,P0004,Groceries,437,0,160,-2.40
141,2022-01-02,S003,P0002,Groceries,175,2,140,-3.40
278,2022-01-03,S004,P0019,Toys,140,1,47,-3.91
511,2022-01-06,S001,P0012,Groceries,59,1,88,-8.37
730,2022-01-08,S002,P0011,Electronics,64,7,79,-2.99
844,2022-01-09,S003,P0005,Furniture,418,4,143,-1.33
1011,2022-01-11,S001,P0012,Toys,205,3,120,-6.05
1107,2022-01-12,S001,P0008,Clothing,432,0,33,-3.55
1149,2022-01-12,S003,P0010,Furniture,74,6,77,-0.11
1180,2022-01-12,S005,P0001,Toys,132,6,84,-2.04


In [38]:
negative_forecast["Units Sold"].describe()

count    673.000000
mean       2.912333
std        2.533651
min        0.000000
25%        1.000000
50%        2.000000
75%        5.000000
max        9.000000
Name: Units Sold, dtype: float64

In [39]:
negative_forecast["Category"].value_counts()

Category
Groceries      143
Toys           140
Clothing       132
Electronics    129
Furniture      129
Name: count, dtype: int64

In [40]:
negative_forecast["Seasonality"].value_counts()

Seasonality
Spring    192
Summer    172
Winter    156
Autumn    153
Name: count, dtype: int64

In [41]:
negative_forecast["Holiday/Promotion"].value_counts()

Holiday/Promotion
0    349
1    324
Name: count, dtype: int64

### Outlier Treatment

The IQR method identified a small number of statistical outliers in
`Units Sold` and `Demand Forecast`.

These observations are retained because statistical outlier status does not
necessarily indicate invalid data. High sales or demand forecasts may
represent legitimate periods of elevated demand.

No observations are removed solely because they are statistical outliers.

In [42]:
negative_forecast["Demand Forecast"].describe()

count    673.000000
mean      -3.693730
std        2.535978
min       -9.990000
25%       -5.470000
50%       -3.380000
75%       -1.540000
max       -0.010000
Name: Demand Forecast, dtype: float64

In [43]:
negative_forecast[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Category",
        "Inventory Level",
        "Units Sold",
        "Units Ordered",
        "Demand Forecast"
    ]
].head(20)

,Date,Store ID,Product ID,Category,Inventory Level,Units Sold,Units Ordered,Demand Forecast
63,2022-01-01,S004,P0004,Groceries,437,0,160,-2.40
141,2022-01-02,S003,P0002,Groceries,175,2,140,-3.40
278,2022-01-03,S004,P0019,Toys,140,1,47,-3.91
511,2022-01-06,S001,P0012,Groceries,59,1,88,-8.37
730,2022-01-08,S002,P0011,Electronics,64,7,79,-2.99
844,2022-01-09,S003,P0005,Furniture,418,4,143,-1.33
1011,2022-01-11,S001,P0012,Toys,205,3,120,-6.05
1107,2022-01-12,S001,P0008,Clothing,432,0,33,-3.55
1149,2022-01-12,S003,P0010,Furniture,74,6,77,-0.11
1180,2022-01-12,S005,P0001,Toys,132,6,84,-2.04


In [44]:
negative_forecast["Category"].value_counts()

Category
Groceries      143
Toys           140
Clothing       132
Electronics    129
Furniture      129
Name: count, dtype: int64

In [45]:
negative_forecast["Seasonality"].value_counts()

Seasonality
Spring    192
Summer    172
Winter    156
Autumn    153
Name: count, dtype: int64

## Cleaning Negative Demand Forecast Values

`Demand Forecast` contains 673 negative observations (0.92% of the
dataset), ranging from -9.99 to -0.01.

Because demand represents an expected quantity, negative forecasts are not
meaningful for inventory analysis. The affected observations are distributed
across categories and seasons rather than being isolated to a specific group.

Since there is no additional information available to reliably reconstruct
the original forecast, negative forecast values are replaced with **0**.

The affected rows are retained because their other fields contain valid
store-product observations.

This treatment is documented as an assumption and should be considered when
interpreting forecast-related analysis.

In [46]:
# Preserve the original dataframe
df_clean = df.copy()

# Count negative forecasts before cleaning
negative_before = (df_clean["Demand Forecast"] < 0).sum()

# Replace negative forecasts with 0
df_clean["Demand Forecast"] = df_clean["Demand Forecast"].clip(lower=0)

print(f"Negative forecasts before cleaning: {negative_before}")
print(
    f"Negative forecasts after cleaning: "
    f"{(df_clean['Demand Forecast'] < 0).sum()}"
)

Negative forecasts before cleaning: 673
Negative forecasts after cleaning: 0


In [47]:
df_clean["Demand Forecast"].describe()

count    73100.000000
mean       141.528727
std        109.209174
min          0.000000
25%         53.670000
50%        113.015000
75%        208.052500
max        518.550000
Name: Demand Forecast, dtype: float64

In [48]:
print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(df_clean):,}")

Rows before cleaning: 73,100
Rows after cleaning:  73,100


In [49]:
duplicate_keys_after = df_clean.duplicated(
    subset=["Date", "Store ID", "Product ID"]
).sum()

print(
    "Duplicate Date + Store ID + Product ID combinations:",
    duplicate_keys_after
)

Duplicate Date + Store ID + Product ID combinations: 0


## Final Data Quality Validation


In [50]:
# Missing values
print("Missing values:")
print(df_clean.isna().sum().sum())

# Duplicate rows
print("\nDuplicate rows:")
print(df_clean.duplicated().sum())

# Duplicate business keys
print("\nDuplicate business keys:")
print(
    df_clean.duplicated(
        subset=["Date", "Store ID", "Product ID"]
    ).sum()
)

# Negative values in key inventory/demand variables
check_columns = [
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand Forecast"
]

print("\nNegative values:")
print((df_clean[check_columns] < 0).sum())

Missing values:
0

Duplicate rows:
0

Duplicate business keys:
0

Negative values:
Inventory Level    0
Units Sold         0
Units Ordered      0
Demand Forecast    0
dtype: int64


In [51]:
OUTPUT_PATH = Path("../data/processed/retail_store_inventory_clean.csv")

df_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")

Cleaned dataset saved to: ..\data\processed\retail_store_inventory_clean.csv


## Data Quality Summary

The following checks were performed on the raw retail inventory dataset:

- Dataset dimensions
- Missing values
- Duplicate rows
- Duplicate `Date + Store ID + Product ID` combinations
- Date validity
- Numerical ranges
- Negative values
- Categorical consistency
- Statistical outliers

The dataset contains 73,100 observations and 15 columns. No missing values
or duplicate rows were identified. The combination of `Date`, `Store ID`, and
`Product ID` is unique across the dataset.

All major numerical variables contain non-negative values except
`Demand Forecast`, which contained 673 negative observations (0.92% of the
dataset). Since negative demand is not meaningful for inventory analysis,
these values were replaced with 0.

Statistical outlier detection identified a small number of observations in
`Units Sold` and `Demand Forecast`. These observations were retained because
an unusually high value does not necessarily indicate an invalid observation.

The categorical fields were also inspected and no obvious inconsistencies
were identified.

In [55]:
quality_summary = pd.DataFrame({
    "Check": [
        "Number of rows",
        "Number of columns",
        "Missing values",
        "Duplicate rows",
        "Duplicate business keys",
        "Invalid dates",
        "Negative inventory",
        "Negative units sold",
        "Negative units ordered",
        "Negative demand forecasts",
        "Unique stores",
        "Unique products",
        "Unique categories",
        "Unique regions",
        "Unique dates"
    ],
    "Result": [
        len(df_clean),
        len(df_clean.columns),
        df_clean.isna().sum().sum(),
        df_clean.duplicated().sum(),
        df_clean.duplicated(
            subset=["Date", "Store ID", "Product ID"]
        ).sum(),
        df_clean["Date"].isna().sum(),
        (df_clean["Inventory Level"] < 0).sum(),
        (df_clean["Units Sold"] < 0).sum(),
        (df_clean["Units Ordered"] < 0).sum(),
        (df_clean["Demand Forecast"] < 0).sum(),
        df_clean["Store ID"].nunique(),
        df_clean["Product ID"].nunique(),
        df_clean["Category"].nunique(),
        df_clean["Region"].nunique(),
        df_clean["Date"].nunique()
    ]
})

quality_summary

,Check,Result
0,Number of rows,73100
1,Number of columns,15
2,Missing values,0
3,Duplicate rows,0
4,Duplicate business keys,0
5,Invalid dates,0
6,Negative inventory,0
7,Negative units sold,0
8,Negative units ordered,0
9,Negative demand forecasts,0


In [59]:
pip install tabulate

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [60]:
quality_summary.to_markdown(
    "../outputs/data_quality_summary.md",
    index=False
)

print("Data quality summary saved.")

Data quality summary saved.


## Initial Observations

The initial data profiling revealed the following:

1. The dataset contains 73,100 daily store-product observations across
   5 stores and 20 products.

2. There are 5 product categories and 4 regions, providing sufficient
   dimensionality for store-, regional-, and category-level comparisons.

3. No missing values or duplicate observations were identified.

4. Each `Date + Store ID + Product ID` combination is unique, supporting
   the use of this combination as the logical business key.

5. `Units Sold` contains zero values, indicating days with no recorded sales.
   These observations are retained.

6. `Discount` ranges from 0 to 20, while `Holiday/Promotion` is binary,
   allowing later analysis of promotional periods.

7. 673 negative `Demand Forecast` values were identified and replaced with
   zero because negative demand is not meaningful for inventory analysis.

8. Statistical outliers were detected in `Units Sold` and `Demand Forecast`,
   but they were retained because they may represent legitimate periods of
   unusually high demand.

Further analysis is required to determine whether inventory levels are
aligned with demand, where low-inventory risk is concentrated, which products
are fast- or slow-moving, and whether apparent overstock conditions exist.

In [61]:
df_clean["Date"] = pd.to_datetime(df_clean["Date"]).dt.strftime("%Y-%m-%d")

In [62]:
df_clean.to_csv(
    "../data/processed/retail_store_inventory_clean.csv",
    index=False
)